In [6]:
import pandas as pd
import os

def compare_csv(file1, file2, output_file):
    # Read CSV files
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)
    
    # Columns that are expected to change
    diff_columns = ["læringsutbytte type", "læringsutbytte"]
    
    # Identify key columns (all columns except the ones that change)
    key_columns = [col for col in df1.columns if col not in diff_columns and col in df2.columns]
    
    if not key_columns:
        print("No common key columns found for merging.")
        return
    
    # Merge dataframes on key columns using an outer join to catch rows only in one file
    merged = pd.merge(df1, df2, on=key_columns, how='outer', suffixes=('_file1', '_file2'), indicator=True)
    
    differences = []
    
    # For rows present in both files, compare the diff_columns values
    both = merged[merged['_merge'] == 'both']
    for idx, row in both.iterrows():
        for col in diff_columns:
            val1 = row.get(f"{col}_file1")
            val2 = row.get(f"{col}_file2")
            # Continue if both values are missing or they are equal
            if pd.isna(val1) and pd.isna(val2):
                continue
            if val1 != val2:
                # Record the differences along with the key columns and the specific column name
                record = {k: row[k] for k in key_columns}
                record["Column"] = col
                record[f"{col}_file1"] = val1
                record[f"{col}_file2"] = val2
                differences.append(record)
    
    # For rows that exist only in file1
    only_in_file1 = merged[merged['_merge'] == 'left_only']
    for idx, row in only_in_file1.iterrows():
        for col in diff_columns:
            record = {k: row[k] for k in key_columns}
            record["Column"] = col
            record[f"{col}_file1"] = row.get(f"{col}_file1")
            record[f"{col}_file2"] = None
            differences.append(record)
    
    # For rows that exist only in file2
    only_in_file2 = merged[merged['_merge'] == 'right_only']
    for idx, row in only_in_file2.iterrows():
        for col in diff_columns:
            record = {k: row[k] for k in key_columns}
            record["Column"] = col
            record[f"{col}_file1"] = None
            record[f"{col}_file2"] = row.get(f"{col}_file2")
            differences.append(record)
    
    # Write the differences to a new CSV file if any differences were found
    if differences:
        diff_df = pd.DataFrame(differences)
        diff_df.to_csv(output_file, index=False)
        print(f"Differences have been written to '{output_file}'.")
    else:
        print("No differences found in the specified columns.")

if __name__ == "__main__":
    # Define folder and file paths
    folder = "compare"
    file1 = os.path.join(folder, "KRISTIANIA_LUBER1.csv")
    file2 = os.path.join(folder, "KRISTIANIA_LUBER2.csv")
    output_file = os.path.join(folder, "differences.csv")
    
    compare_csv(file1, file2, output_file)

Differences have been written to 'compare/differences.csv'.
